In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report, r2_score, mean_squared_error
import joblib


In [ ]:
linear_df = pd.read_csv("../aiml_salary.csv")
binary_df = pd.read_csv("Heart_Dataset.csv")

In [ ]:
print("Dataset for linear model : ")
print(linear_df.head(2))
print("\nDataset for classification model : ")
print(binary_df.head(2))


In [ ]:
cols_to_drop = ["job_id","job_title","industry","experience_level","city","posting_year",
                "demand_growth_yoy_pct","posting_month","is_remote_friendly","is_llm_role",
                "ai_salary_premium_pct","remote_work","company_size","benefits_score_10",
                "salary_tier","country","demand_score"]
linear_df = linear_df.drop(cols_to_drop, axis=1, errors='ignore')

linear_y = linear_df["annual_salary_usd"]
linear_X = linear_df.drop(["annual_salary_usd","salary_min_usd","salary_max_usd"], axis=1, errors='ignore')


In [ ]:
#One hot encodding
linear_cat_cols = ["job_category", "education_required", "required_skills"]
linear_X = pd.get_dummies(data = linear_X, columns=linear_cat_cols, dtype=int, drop_first=True)

# Scaling
linear_num_cols = ["years_of_experience"]
linear_scaler = StandardScaler()
linear_X[linear_num_cols] = linear_scaler.fit_transform(linear_X[linear_num_cols])

In [ ]:
X_train_lin, X_test_lin, y_train_lin, y_test_lin = train_test_split(linear_X, linear_y, test_size=0.2, random_state=42)

lin_model = LinearRegression()
lin_model.fit(X_train_lin, y_train_lin)
y_pred_lin = lin_model.predict(X_test_lin)

In [ ]:
print("R2 Score: ", r2_score(y_test_lin, y_pred_lin))
print("RMSE: ", np.sqrt(mean_squared_error(y_test_lin, y_pred_lin)))
joblib.dump(lin_model, "linear_salary_model.joblib")
joblib.dump(linear_scaler, "linear_scaler.joblib")
joblib.dump(linear_X.columns.tolist(), "linear_columns.joblib") 


In [ ]:
df = binary_df.copy()
df = df.drop(["FastingBS","RestingECG"], axis=1)

X = df.drop(["HeartDisease"], axis=1)
y = df["HeartDisease"]

numerical = ["Age","RestingBP","Cholesterol","MaxHR","Oldpeak"]
categorical = ["Sex","ChestPainType","ExerciseAngina","ST_Slope"]


In [ ]:

for col in numerical:
    X[col] = pd.to_numeric(X[col], errors="coerce")
    X[col] = X[col].replace(0, np.nan)
X[numerical] = X[numerical].fillna(X[numerical].mean())
for col in categorical:
    X[col] = X[col].fillna(X[col].mode()[0])



In [ ]:
#Scaling 
scaler = StandardScaler()
num_scaled = pd.DataFrame(scaler.fit_transform(X[numerical]), columns=numerical, index=X.index)

#One hot encodding 
cat_encoded = pd.get_dummies(X[categorical], drop_first=True).astype(int)

In [ ]:
X = pd.concat([num_scaled, cat_encoded], axis=1)

print("\nFinal X Shape :", X.shape)
print("Any NaN Left :", X.isnull().sum().sum())

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42)

print("X_train Shape :", X_train.shape)
print("X_test Shape :", X_test.shape)

In [ ]:
models= {
"Logistic_model" : LogisticRegression(max_iter=100),
"KNN_model" : KNeighborsClassifier(n_neighbors=5),
"Naive_model" :GaussianNB()
}


In [ ]:
results={}
for name,model in models.items():
    model.fit(X_train,y_train)
    y_pred = model.predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    results[name] = acc

    print(f"\n{name}")
    print("Accuracy :", round(acc*100, 2), "%")
    print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))
    print("Classification Report:\n", classification_report(y_test, y_pred))
    print("-"*50)
    joblib.dump(model, f"heart_{name.lower().replace(' ', '_')}.joblib")


In [ ]:
best_model_name = max(results, key=results.get)
print(f"\nBest Model: {best_model_name} with Accuracy: {results[best_model_name]*100:.2f}%")

In [ ]:
joblib.dump(scaler, "heart_scaler.joblib")
joblib.dump(X.columns.tolist(), "heart_columns.joblib")

print("\nFiles Saved Successfully: scaler, columns, and all 3 models")